In [1]:
import os
import sys

# Move up one directory to the project root and add it to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Now your imports will work perfectly!
from src.data_loader import load_insurance_data, calculate_insurance_metrics
from src.eda_utils import get_missing_summary, plot_loss_ratio_by_dimension

In [2]:
# In your notebook cell:
from src.data_loader import load_insurance_data, calculate_insurance_metrics

# Point directly to your zipped text file path
zip_path = '../data/MachineLearningRating_v3.zip' 

df = load_insurance_data(zip_path)
df = calculate_insurance_metrics(df)

print(f"Data Successfully Ingested! Shape: {df.shape}")
print("Columns Loaded:", df.columns.tolist()[:5], "... total columns:", len(df.columns))


c:\Users\Almazt\OneDrive - Ethiopian Airlines\Desktop\Collection\10 Academy\week 3 insurance-risk-analytics\src\data_loader.py:32: DtypeWarning: Columns (0: CapitalOutstanding, 1: CrossBorder) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(z.open(target_internal_file), sep=separator)


Data Successfully Ingested! Shape: (1000098, 54)
Columns Loaded: ['UnderwrittenCoverID', 'PolicyID', 'TransactionMonth', 'IsVATRegistered', 'Citizenship'] ... total columns: 54


In [3]:
# Notebook Cell Execution
from src.data_loader import load_insurance_data, calculate_insurance_metrics
from src.eda_utils import get_missing_summary, plot_loss_ratio_by_dimension

# Point directly to your zipped text file path
zip_path = '../data/MachineLearningRating_v3.zip' 

df = load_insurance_data(zip_path)
# df = load_insurance_data('../data/insurance_data.csv')
df = calculate_insurance_metrics(df)

# Portfolio Wide Loss Ratio
portfolio_loss_ratio = df['TotalClaims'].sum() / df['TotalPremium'].sum()
print(f"Global Portfolio Loss Ratio: {portfolio_loss_ratio:.2%}")


c:\Users\Almazt\OneDrive - Ethiopian Airlines\Desktop\Collection\10 Academy\week 3 insurance-risk-analytics\src\data_loader.py:32: DtypeWarning: Columns (0: CapitalOutstanding, 1: CrossBorder) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(z.open(target_internal_file), sep=separator)


Global Portfolio Loss Ratio: 104.77%


In [ ]:
# working through a data analysis notebook—likely focused on short-term car or property insurance data, given the mention of Gauteng (South Africa), premiums, and claims.

# These three notes outline exactly how to handle the heavy-tailed, messy nature of insurance data. Let's break down how to actually write the Python code to answer these questions in your notebook.

# 1. Portfolio Loss Ratio Distribution
# The Loss Ratio is the ultimate health metric for an insurance portfolio. It tells you how many cents of every premium dollar are being paid out as claims.

# Loss Ratio= 
# Total Premium
# Total Claims
# ​
 
# First, you need to calculate the baseline for the entire dataset, and then look at it by Province to spot the high-risk areas.

import pandas as pd

# Point directly to your zipped text file path
zip_path = '../data/MachineLearningRating_v3.zip' 

df = load_insurance_data(zip_path)

# df = pd.read_csv('your_data.csv')  # Or read_excel, etc.

# 1. Calculate the overall baseline loss ratio
total_claims = df['TotalClaims'].sum()
total_premium = df['TotalPremium'].sum()
baseline_loss_ratio = total_claims / total_premium

print(f"Overall Portfolio Baseline Loss Ratio: {baseline_loss_ratio:.2%}")


In [ ]:

# 2. Isolate geographical risk outliers using your custom function
# Look closely at Gauteng vs. rural provinces in the resulting plot
import pandas as pd

# Point directly to your zipped text file path
zip_path = '../data/MachineLearningRating_v3.zip' 

df = load_insurance_data(zip_path)

plot_loss_ratio_by_dimension(df, 'Province')

# What to look for: If your baseline is 65%, but Gauteng is sitting at 85%, that confirms the note's hypothesis: dense urban traffic and higher accident rates in Gauteng are driving structural risk.

In [ ]:
# 2. Financial Variable OutliersInsurance claims are notoriously heavy-tailed (log-normal). Most people claim $\$0$ or very little, but a few total-loss accidents create massive spikes. As your note says, don't drop them! Instead, let's visualize them and then fix them using the two suggested methods.Step A: Visualize the OutliersPythonimport matplotlib.pyplot as plt
import seaborn as sns

# Plot boxplots to see the extreme right tails
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(data=df, y='TotalClaims', ax=axes[0])
axes[0].set_title('Outliers in Total Claims')

sns.boxplot(data=df, y='CustomValueEstimate', ax=axes[1])
axes[1].set_title('Outliers in Custom Value Estimate')

plt.tight_layout()
plt.show()

In [ ]:
# Step B: Handle the Outliers (Choose One Method)
# Depending on what your notebook requires next, you can either cap the extreme values or log-transform them.

# OPTION 1: Cap at the 99th percentile (Winsorization)
cap_claims = df['TotalClaims'].quantile(0.99)
cap_value = df['CustomValueEstimate'].quantile(0.99)

df['TotalClaims_Capped'] = df['TotalClaims'].clip(upper=cap_claims)
df['CustomValueEstimate_Capped'] = df['CustomValueEstimate'].clip(upper=cap_value)

# OPTION 2: Log Transformation (Smooths out the heavy tail for modeling)
import numpy as np

df['TotalClaims_Log'] = np.log1p(df['TotalClaims']) # log(x + 1)
df['CustomValueEstimate_Log'] = np.log1p(df['CustomValueEstimate'])

In [ ]:
# 3. Temporal (Time-Series) TrendsTo see if macro factors (like inflation) or seasonal trends (like summer hail storms in Gauteng) are spiking claims, you need to aggregate the data by month.Python# 1. Group dynamically by month and sum the financials
monthly_trends = df.groupby(df['TransactionMonth'].dt.to_period('M'))[['TotalClaims', 'TotalPremium']].sum()

# 2. Calculate the monthly loss ratio to see if spikes match specific months
monthly_trends['MonthlyLossRatio'] = monthly_trends['TotalClaims'] / monthly_trends['TotalPremium']

# 3. Plot the timeline to visually check for structural deviations
monthly_trends['MonthlyLossRatio'].plot(kind='line', marker='o', figsize=(10, 5))
plt.title('18-Month Loss Ratio Trend')
plt.ylabel('Loss Ratio')
plt.xlabel('Transaction Month')
plt.grid(True)
plt.show()

# Display the raw numbers
print(monthly_trends)
# What to look for: Look at the MonthlyLossRatio columns over the 18-month timeline. Is there a random, massive spike in one or two specific months? If yes, look up historical weather or economic events for that period to explain the "structural deviation."

In [ ]:
# Here is a clean, robust way to write that plot_loss_ratio_by_dimension function.

# To make it truly useful for your analysis, the function will calculate the group-level loss ratios, sort them from highest risk to lowest risk, and plot a clear visual baseline so you can immediately spot which provinces are the outliers.

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def plot_loss_ratio_by_dimension(df, dimension):
    """
    Groups data by a categorical dimension, calculates the loss ratio for each group,
    and plots a sorted bar chart against the overall portfolio baseline.
    """
    # 1. Calculate the overall portfolio baseline
    overall_baseline = df['TotalClaims'].sum() / df['TotalPremium'].sum()
    
    # 2. Group by the dimension and sum the financials
    grouped = df.groupby(dimension)[['TotalClaims', 'TotalPremium']].sum()
    
    # 3. Calculate the loss ratio for each group
    grouped['LossRatio'] = grouped['TotalClaims'] / grouped['TotalPremium']
    
    # 4. Sort from highest risk to lowest risk for easier analysis
    grouped = grouped.sort_values(by='LossRatio', ascending=False).reset_index()
    
    # 5. Create the plot
    plt.figure(figsize=(10, 6))
    
    # Custom palette: highlights bars that cross the baseline
    colors = ['#d9534f' if lr > overall_baseline else '#5cb85c' for lr in grouped['LossRatio']]
    
    sns.barplot(
        data=grouped, 
        x=dimension, 
        y='LossRatio', 
        palette=colors,
        hue=dimension,  # Keeps seaborn happy
        legend=False
    )
    
    # 6. Draw the overall portfolio baseline line
    plt.axhline(
        y=overall_baseline, 
        color='black', 
        linestyle='--', 
        linewidth=2, 
        label=f'Portfolio Baseline ({overall_baseline:.1%})'
    )
    
    # 7. Formatting and labels
    plt.title(f'Loss Ratio Distribution by {dimension}', fontsize=14, pad=15)
    plt.ylabel('Loss Ratio (Claims / Premium)', fontsize=12)
    plt.xlabel(dimension, fontsize=12)
    plt.xticks(rotation=45, ha='right')
    
    # Change y-axis to percentage format for clarity
    current_values = plt.gca().get_yticks()
    plt.gca().set_yticklabels([f'{x:.0%}' for x in current_values])
    
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.show()
    
    # Optional: return the summary dataframe for quick text inspection
    return grouped[[dimension, 'LossRatio']]

In [ ]:
# How to use it:
# Once you run the block of code above to define the function, you can call it on any categorical column in your dataset:

# To answer your specific notebook question:
summary_df = plot_loss_ratio_by_dimension(df, 'Province')

# You can also use it to check other dimensions later, like VehicleType!
# plot_loss_ratio_by_dimension(df, 'VehicleType')
# Why this design works perfectly for your notebook note:
# The Sorting: Sorting the provinces from highest to lowest forces the worst "geographical risk outliers" (like Gauteng) straight to the left side of the chart.

# The Visual Cue: The bars dynamically change color. Any province with a loss ratio higher than the total portfolio baseline turns red, while safer corridors stay green.